In [ ]:
!pip install ultralytics

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import imutils
from imutils import contours
from ultralytics import YOLO


In [ ]:
def is_id(im_path):
  model = YOLO('/content/drive/MyDrive/ColabNotebooks/id_info_model.pt')
  results = model(im_path)
  for r in results:
      data_np = r.boxes.data.cpu().numpy()
      cls_np = r.boxes.cls.cpu().numpy()
      conf_np = r.boxes.conf.cpu().numpy()
      filtered_data = (data_np[cls_np ==2],conf_np[cls_np ==2])
      if len(filtered_data[0])>0:
        if filtered_data[1][0]>0.5:
          return True
        else:
          return False
      else:
          return False

# is_id("/content/output_image.jpg")

In [ ]:
# model = YOLO('/content/drive/MyDrive/ColabNotebooks/id_info_model.pt')
# model.predict('/content/id1.jpg',save=True)

In [ ]:
# names={0: 'Address', 1: 'Birth', 2: 'Factory', 3: 'Gender', 4: 'HusName', 5: 'ID_B', 6: 'ID_F', 7: 'Name', 8: 'Occup', 9: 'Rel', 10: 'Start', 11: 'Status', 12: 'end'}
def extract_text(img_path):
  model = YOLO('/content/drive/MyDrive/ColabNotebooks/id_info_model.pt')
  results = model(img_path)
  img=cv2.imread(img_path)
  bluerd_img=cv2.imread(img_path)
  for r in results:
    data_np = r.boxes.data.cpu().numpy()
    cls_np = r.boxes.cls.cpu().numpy()

    target_class = [7,0,6]

    filtered_data = data_np[cls_np == target_class[0]]
    filtered_data2 = data_np[cls_np == target_class[1]]
    filtered_data3 = data_np[cls_np == target_class[2]]

    bounding_boxes = filtered_data[:, :4]
    bounding_boxes2 = filtered_data2[:, :4]
    bounding_boxes3 = filtered_data3[:, :4]
    f_name=img[int(bounding_boxes[0][1])-10:int(bounding_boxes[0][3])-33,int(bounding_boxes[0][0])+30:int(bounding_boxes[0][2])+10]
    rest_name=img[int(bounding_boxes[0][1])+32:int(bounding_boxes[0][3])+6,int(bounding_boxes[0][0])-30:int(bounding_boxes[0][2])+10]
    address= img[int(bounding_boxes2[0][1])-6:int(bounding_boxes2[0][3])-30,int(bounding_boxes2[0][0])-30:int(bounding_boxes2[0][2])+10]
    regin= img[int(bounding_boxes2[0][1])+25:int(bounding_boxes2[0][3])+6,int(bounding_boxes2[0][0])-30:int(bounding_boxes2[0][2])+10]
    id=img[int(bounding_boxes3[0][1])-15:int(bounding_boxes3[0][3])+6,int(bounding_boxes3[0][0])-10:int(bounding_boxes3[0][2])+10]


    bluerd_img[int(bounding_boxes3[0][1])-15:int(bounding_boxes3[0][3])+6,int(bounding_boxes3[0][0])-10:int(bounding_boxes3[0][2])+10]=cv2.GaussianBlur(bluerd_img[int(bounding_boxes3[0][1])-15:int(bounding_boxes3[0][3])+6,int(bounding_boxes3[0][0])-10:int(bounding_boxes3[0][2])+10], (51, 51), 0)
    bluerd_img[int(bounding_boxes[0][1])+32:int(bounding_boxes[0][3])+6,int(bounding_boxes[0][0])-30:int(bounding_boxes[0][2])+10]=cv2.GaussianBlur(bluerd_img[int(bounding_boxes[0][1])+32:int(bounding_boxes[0][3])+6,int(bounding_boxes[0][0])-30:int(bounding_boxes[0][2])+10], (51, 51), 0)

    # name=img[int(bounding_boxes[0][1]):int(bounding_boxes[0][3]),int(bounding_boxes[0][0]):int(bounding_boxes[0][2])]
    # address= img[int(bounding_boxes2[0][1]):int(bounding_boxes2[0][3]),int(bounding_boxes2[0][0]):int(bounding_boxes2[0][2])]
    # return [name,address,id]

    cv2.imwrite("bluerd_img.jpg",bluerd_img)
    return [f_name,rest_name,address,id,regin]


res=extract_text('/content/id.jpg')

# cv2_imshow(res[0])
# cv2_imshow(res[1])
# cv2_imshow(res[2])
# cv2_imshow(res[3])
# cv2_imshow(res[4])

In [ ]:
def get_numbers_from_img(id:np.array):
  # id=cv2.cvtColor(id,cv2.COLOR_BGR2GRAY)
  ref = cv2.threshold(id, 0, 255, cv2.THRESH_BINARY_INV|cv2.THRESH_OTSU)[1]
  ref=cv2.GaussianBlur(ref, (3, 3), 0)
  kernal = np.ones((2,2), np.uint8)
  ref = cv2.dilate(ref, kernal, iterations=1)
  # plt.imshow(ref,cmap="gray")
  # plt.show()
  refCnts = cv2.findContours(ref.copy(), cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
  refCnts = imutils.grab_contours(refCnts)
  refCnts = contours.sort_contours(refCnts, method="left-to-right")[0]
  # print(len(refCnts))
  digits = {}
  i=0
  for  c in refCnts:
      (x, y, w, h) = cv2.boundingRect(c)
      roi = id[y-3:y + h+3, x-3:x + w+3]
      # cv2.rectangle(id,(x-6,y-6),(x + w+6,y + h+6),(0,255,0),2)
      if len(roi)>0:
        roi = cv2.resize(roi, (57, 88))
        roi=cv2.bilateralFilter(roi,11,17,17)
        digits[i] = roi
        # plt.imshow(roi,cmap="gray")
        # plt.axis('off')
        # plt.show()
        i+=1
  return digits

# get_numbers_from_img(res[3])

In [ ]:
def get_number(id_part)-> list:
  """
  A Function that extract the numbers from the ID part from image as intger values.

  Parameters:
  - the National ID card image path.

  Returns:
  - it return a List the numbers that been extracted from image and the list with type intger.
  """
  # img2=cv2.imread(img_path)
  # img2 = cv2.resize(id_part, (700, 480))
  gray2=cv2.cvtColor(id_part,cv2.COLOR_BGR2GRAY)
  # id = gray2[360:430 , 300:]
  id=gray2

  template0=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_0.png",0)
  template0 = cv2.resize(template0, (57, 88))
  template0=cv2.bilateralFilter(template0,11,17,17)



  template1=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_1.png",0)
  template1 = cv2.resize(template1, (57, 88))
  template1=cv2.bilateralFilter(template1,11,17,17)

  template2=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_two.png",0)
  template2 = cv2.resize(template2, (57, 88))
  template2=cv2.bilateralFilter(template2,11,17,17)

  template3=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_3.png",0)
  template3 = cv2.resize(template3, (57, 88))
  template3=cv2.bilateralFilter(template3,11,17,17)


  template4=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_4.png",0)
  template4 = cv2.resize(template4, (57, 88))
  template4=cv2.bilateralFilter(template4,11,17,17)


  template5=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_5.png",0)#فيها مشاكل
  template5 = cv2.resize(template5, (57, 88))
  template5=cv2.bilateralFilter(template5,11,17,17)

  template6=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_6.png",0)
  template6 = cv2.resize(template6, (57, 88))
  template6=cv2.bilateralFilter(template6,11,17,17)

  template7=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_7.png",0)
  template7 = cv2.resize(template7, (57, 88))
  template7=cv2.bilateralFilter(template7,11,17,17)

  template8=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_8.png",0)
  template8 = cv2.resize(template8, (57, 88))
  template8=cv2.bilateralFilter(template8,11,17,17)

  template9=cv2.imread("/content/drive/MyDrive/ColabNotebooks/arabic_numbers/number_9.png",0)
  template9 = cv2.resize(template9, (57, 88))
  template9=cv2.bilateralFilter(template9,11,17,17)

  num_templates=[template0,template1,template2,template3,template4,template5,template6,template7,template8,template9]

  dig=get_numbers_from_img(id)
  numbers=[]
  for num in range(0,len(dig)):
    # print((w,h),template4.shape)
    for number in range(0,10):
     w,h=num_templates[number].shape[::-1]
     res=cv2.matchTemplate(dig[num],num_templates[number],cv2.TM_CCORR_NORMED)
     thresh=0.999
     loc=np.where(res>=thresh)
     if len(loc[0]) >0 and len(loc[1])>0:
      # print(number)
      numbers.append(number)
      break
     elif len(loc[0])==0 and len(loc[1])==0 and number==9:
      loc2=([],[])
      thresh2=thresh-0.001
      while len(loc2[0])==0 and len(loc2[1])==0:
        for number2 in range(0,10):
          #  w,h=num_templates[number2].shape[::-1]
           res2=cv2.matchTemplate(dig[num],num_templates[number2],cv2.TM_CCORR_NORMED)
           loc2=np.where(res2>=thresh2)
           if len(loc2[0]) >0 and len(loc2[1])>0:
            numbers.append(number2)
            break
           elif len(loc2[0])==0 and len(loc2[1])==0 and number2==9:
                thresh2-=0.001

  numbers=numbers[0:14]
  return numbers

get_number(res[3])

In [ ]:
import datetime as dt

class ID:
    def __init__(self,number:int):

        self.Number=number

    def get_BirthDate(self):
        """
         A Function that extract the brith date from the National ID card number.

         Parameters:
         - the id card number with type intger.

         Returns:
         - it return BirthDate of the id card owner with type datetime.
        """
        centray=""
        Birthyear=""
        self.Number=str(self.Number)
        if self.Number[0]=="2":
            centray="1900"
        elif self.Number[0]=="3":
            centray="2000"
        Birthyear=int(centray)+int(self.Number[1:3])
        Month=int(self.Number[3:5])
        Day=int(self.Number[5:7])
        BirthDate=dt.datetime(Birthyear,Month,Day)
        return BirthDate

    def get_BirthPlace(self):
        """
         A Function that extract the Birth Place from the National ID card number.

         Parameters:
         - the id card number with type intger.

         Returns:
         - it return Birth Place of the id card owner with type string.
        """
        self.Number=str(self.Number)
        BirthPlace_num=self.Number[7:9]
        Governorates=["Cairo","Alexandria","Port Said","Suez","Damietta","Dakahlia","elsharkia","Qalyubia","Kafr El-Sheikh","elgharbia","Menoufia","elbehera","Ismailia","Giza","Bani Sweif","Fayoum","Minya","Asyut","Sahaj","Qena","Aswan","Luxor","The Red Sea","the new Valley","matrooh","North Sinai","South of Sinaa","out of Egypt"]
        ara_Governorates=["القاهرة","الأسكندرية","بور سعيد","السويس","دمياط","الدقهلية","الشرقية","القليوبية","كفر الشيخ","الغربية","المنوفية","البحيرة","الاسماعيلية","الجيزة","بني سويف","الفيوم","المنيا","أسيوط","سوهاج","قنا","اسوان","الأقصر","البحر الأحمر","الوادي الجديد","مطروح","شمال سيناء","جنوب سيناء","خارج مصر"]
        G_Numbers=[1,2,3,4]
        FinalDict={}

        for i in range(11,36):
            if i==30 or i==20:
                continue
            G_Numbers.append(i)

        G_Numbers.append(88)
        x=0
        for i in G_Numbers:
            FinalDict[i]=ara_Governorates[x]
            x=x+1

        BirthPlace_num=int(BirthPlace_num)
        return FinalDict[BirthPlace_num]


    def get_Gender(self):
        """
         A Function that extract the Gender from the National ID card number.

         Parameters:
         - the id card number with type intger.

         Returns:
         - it return Gender of the id card owner with type string (male,female).
        """
        self.Number=str(self.Number)
        Gender=self.Number[len(self.Number)-2]
        Gender=int(Gender)
        if Gender%2==0:
            return "Female"
        else:
            return "Male"





def Age(BirthDate:dt.datetime) -> tuple:
    """
         A Function that Calculate the Age of specific BirthDate.

         Parameters:
         - A BirthDate of the id card owner with type datetime.

         Returns:
         - it return tuble of the with type int the id owner age in Years and how much Months.
    """
    now=dt.datetime.today()
    if now.month>=BirthDate.month:
        M=now.month-BirthDate.month
    else:
       M=(now.month-BirthDate.month)+12
    d=now.day-BirthDate.day
    Days=(now-BirthDate).days
    if d<0:
        F=30
        if BirthDate.month==1 or BirthDate.month==3 or BirthDate.month==5 or BirthDate.month==7 or BirthDate.month==8 or BirthDate.month==10 or BirthDate.month==12:
            F=31
        if BirthDate.month==2 :
            F=28
        d=d+F
    Years=Days//365
    return (Years,M)

In [ ]:
# !python3 -m pip install paddlepaddle
# !pip install "paddleocr>=2.0.6"

In [ ]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='ar')

In [ ]:
def confert_to_eng(id):
  for i in id:
    if i=="٠":
      id=id.replace("٠","0")
    if i==" ":
      id=id.replace(" ","")
    if i=="١":
      id=id.replace("١","1")
    if i=="٢":
      id=id.replace("٢","2")
    if i=="٣":
      id=id.replace("٣","3")
    if i=="٤":
      id=id.replace("٤","4")
    if i=="٥":
      id=id.replace("٥","5")
    if i=="٦":
      id=id.replace("٦","6")
    if i=="٧":
      id=id.replace("٧","7")
    if i=="٨":
      id=id.replace("٨","8")
    if i=="٩":
      id=id.replace("٩","9")
  return id[::-1]


In [ ]:
import cv2
import re

def revers_txt(txts):
    """
    Reverses the order of text based on the boxes.
    """
    txts.sort(key=lambda x: x[1], reverse=True)
    return txts

def get_data_paddelOCR(images_list,read_id=False)-> dict:
    """
    A Function that extract the text from image or specific part from image using paddleOCR.

    Parameters:
    - images_list: List of images containing text.

    Returns:
    - A dictionary with extracted text and corresponding data.
    """
    data=[]
    id_data=[]
    if read_id==True:
      id = images_list[3]
      id = cv2.resize(id, (1080, 350))
      data.append('id')
      id_data.append(id)
    f_name = images_list[0]
    f_name = cv2.resize(f_name, (1080, 350))
    f_name=cv2.bilateralFilter(f_name,11,17,17)
    l_name = images_list[1]
    l_name = cv2.resize(l_name, (1080, 350))
    l_name=cv2.bilateralFilter(l_name,11,17,17)
    address = images_list[2]
    address = cv2.resize(address, (1080, 350))
    address=cv2.bilateralFilter(address,11,17,17)
    regin = images_list[4]
    regin = cv2.resize(regin, (1080, 350))
    regin=cv2.bilateralFilter(regin,11,17,17)
    data.append('f_name')
    data.append('l_name')
    data.append('address')
    data.append('regin')
    id_data.append(f_name)
    id_data.append(l_name)
    id_data.append(address)
    id_data.append(regin)


    data_out={}

    for x,y in zip(id_data,data):
        text = ocr.ocr(x, cls=True)
        data_out[y]=text

    boxes = {}
    txts = {}

    for key,item in data_out.items():
        if data_out[key][0]==None:
            continue
        txts[key] = []
        boxes[key] = []
        for i in range(len(item[0])):
            boxes[key].append(item[0][i][0][0])
            txts[key].append((item[0][i][1][0], item[0][i][0][0]))

    new_data_out={}

    for key,item in txts.items():
        new_data_out[key]=revers_txt(item)

    output_data = {}
    for key,item in new_data_out.items():
        output_data[key] = " ".join([i[0] for i in item])

    for key,item in output_data.items():
      if key=="id":
        continue
      words = item.split()
      reversed_words = words[::-1]
      reversed_text = " ".join(reversed_words)[::-1]
      output_data[key]=reversed_text
    if read_id==True:
      output_data["id"]=confert_to_eng(output_data["id"])
    return output_data

# output = get_data_paddelOCR(res,False)
# print(output)

In [ ]:
def get_final_data(im_path) -> dict:
  try:
    """
      A function named 'get_final_data' that processes an image to extract various information, including personal identification numbers, birth date, birth place, gender, and age.

      Parameters:
      - img_path: The path to the input image file.

      Returns:
      - A dictionary containing the extracted information from the image, including personal identification numbers, birth date, birth place, gender, and age. The function utilizes the 'get_data' and 'get_number' functions to retrieve initial data and identification numbers. It then processes and validates the obtained ID numbers, extracts birth date, and calculates age.
    """
    res=extract_text(im_path)
    data=get_data_paddelOCR(res)
    new_data={}
    new_data["id"]=get_number(res[3])
    for key,item in data.items():
      new_data[key]=item
    id=""
    for key,value in new_data.items():
      if key=="id":
        for i in value:
          id+=str(i)
    Month=int(id[3:5])
    Day=int(id[5:7])
    if Month>12 or Month==0:
      id=id.replace(id[3:5], "12")
      new_data['problem']='Date of birth and age will not be correct'
    if Day>31 or Day==0:
      id=id.replace(id[5:7], "29")
      new_data['problem']='Date of birth and age will not be correct'
    new_data["id"]=id
    x=ID(int(id))
    x1=x.get_BirthDate()
    try:
      x2=x.get_BirthPlace()
    except Exception as e:
      x2="Unknown"
    x3=x.get_Gender()
    if x3=="Female":
      x3="انثى"
    elif x3=="Male":
      x3="ذكر"
    age=Age(x1)
    new_data["Birth Date"]=x1.strftime("%Y-%m-%d")
    new_data["Birth Place"]=x2
    new_data["Gender"]=x3
    new_data["Age"]= f"{age[0]} year and {age[1]} month"
    return new_data
  except Exception as e:
    try:
      res=extract_text(im_path)
      data=get_data_paddelOCR(res,True)
      new_data={}
      for key,item in data.items():
        new_data[key]=item
      new_data['problem']='mostly id will not be correct'
      return new_data
    except Exception as e:
      return {"problem":"Please make sure that the entered photo is a national ID card, or make sure that the card is placed in the photo correctly"}

get_final_data("/content/id.jpg")